### Convolutional neural networks 

MNIST classifier

In [ ]:
import pandas as pd
import numpy as np
from pynq import Overlay, allocate
from utils import *

Load overlay

In [ ]:
# Load the FPGA hardware design (bitstream + metadata).
# The file "bd_wrapper_mnist.xsa" was generated by Vivado and contains:
#   - The PL bitstream
#   - Hardware handoff information (address map, IP configuration)
#   - AXI interconnect descriptions used by PYNQ
#
# Loading the Overlay configures the programmable logic (PL) and
# exposes the hardware IP blocks as Python-accessible objects.

ol = Overlay("bd_wrapper_mnist.xsa")

Explore available Ip cores

In [ ]:
# List all IP cores available in the loaded overlay.
# 'ip_dict' is a dictionary autogenerated by PYNQ, where:
#   - keys   = IP core names (as synthesized in Vivado)
#   - values = metadata (address ranges, driver info, registers, etc.)
#
# This is useful for discovering the names of DMA blocks, peripherals, and custom accelerators.

list(ol.ip_dict.keys())

Create objects

In [ ]:
# Get a reference to the AXI DMA instance defined in the hardware design.
# The name 'axi_dma_0' must match the IP name inside the Vivado block design.
dma = ol.axi_dma_0

# DMA channels:
#   - sendchannel:  transfers data from PS to PL  (input to accelerator)
#   - recvchannel:  transfers data from PL to PS  (output from accelerator)
dma_send = dma.sendchannel
dma_recv = dma.recvchannel

# Get a reference to the custom accelerator IP core.
# 'myproject_mnist_accel_0' is the name Vivado generated for the HLS module.
# This gives access to its AXI-Lite control registers (e.g., start, status).
ip = ol.myproject_mnist_accel_0

IP_CTRL_REG = 0x00  # Control register: writing 1 triggers inference

Load testing dataset

MNIST is stored in a csv file, where the images are normalized.

In [ ]:
# Load the MNIST dataset from CSV.
# The file format is:
#   - Each row represents a flattened 28x28 image (784 pixel values)
#   - The last column contains the digit label (0–9)

data = pd.read_csv("test_dataset/mnist_dataset.csv").values

# Split into pixel data (features) and labels (targets).
# pixels : shape (N, 784), grayscale values in range 0–255
# labels : shape (N,), integer class labels
pixels = data[:, :-1]
labels = data[:, -1].astype(int)

# Number of samples in the dataset.
N = pixels.shape[0]


Define DMA buffers

In [ ]:
# Allocate DMA-compatible input buffer for the MNIST accelerator.
# - MNIST images are 28×28 = 784 pixels.
# - Each pixel will be converted to the accelerator's AXIS format
#   (commonly packed into a 32-bit word, hence uint32).
in_buffer = allocate(shape=(784,), dtype=np.uint32)

# Allocate DMA-compatible o§utput buffer.
# - The accelerator outputs 10 values (one per digit class: 0–9).
# - We use uint32 to match the AXIS interface and the accelerator’s output format.
out_buffer = allocate(shape=(10,), dtype=np.uint32)


Inference function

In [ ]:
def inference_image(img):
    
    """
    Perform inference on a single 28x28 image using the hardware accelerator.

    Parameters
    ----------
    img : array (length 784)
        Flattened grayscale image. Each value is expected to be a float.

    Returns
    -------
    scores : list of float
        The output scores for the 10 classes returned by the hardware IP.
        Values are converted from ap_fixed<16,12> format.
    """
    
    # Fill the DMA input buffer with the converted image values.
    # Each pixel is converted from float to AXI word formatted as ap_fixed<16,12>.
    for i in range(784):
        in_buffer[i] = float_to_axis_word(img[i])

    # Start the inference IP core.
    # Register 0x00 is typically the control register; writing 1 triggers execution.
    ip.write(IP_CTRL_REG, 1)

    # Prepare DMA to receive the output (device → host).
    dma_recv.transfer(out_buffer)

    # Send the input buffer to the IP (host → device).
    dma_send.transfer(in_buffer)

    # Wait for both DMA transfers to complete.
    dma_send.wait()
    dma_recv.wait()

    # Convert raw output words (ap_fixed<16,12>) into Python floats.
    #  - 16 bits total
    #  - 12 integer bits
    #  - 4 fractional bits, divide by 2^4
    scores = []
    for i in range(10):
        raw = int(out_buffer[i] & 0xFFFF)  # isolate 16-bit value

        # Manual sign extension for 16-bit signed numbers
        if raw & 0x8000:   # if sign bit is set
            raw -= 0x10000

        # Convert fixed-point to float
        scores.append(raw / (1 << 4))  # divide by 2^4

    return scores


Execute the inferece process and save predictions 

In [ ]:
correct = 0
preds = []

for i, (img, label) in enumerate(zip(pixels, labels)):
    # Run inference on the FPGA accelerator.
    # Returns an array of 10 scores (one per class 0–9).
    scores = inference_image(img)
    # Predicted digit = index of the largest output score.
    pred = int(np.argmax(scores))
    preds.append(pred)
    # Update accuracy counter.
    if pred == label:
        correct += 1
    # Progress log every 1000 samples.
    if i % 1000 == 0:
        print(f"{i}/{len(pixels)} processed...")

accuracy = correct / len(pixels)
print("Accuracy:", accuracy)

Compute and print confusion matrix

In [ ]:
# Compute the confusion matrix for MNIST classification.
#   labels : ground-truth digit labels (0–9)
#   preds  : predicted digit labels (0–9)
#   10     : number of classes
#
# The resulting matrix is 10×10 where:
#   row = true label
#   col = predicted label
cm = confusion_matrix_np(labels, preds, 10)


In [ ]:
plot_confusion_matrix(cm, class_names=range(10))

----

This work was supported in part by the [AMD University Program](https://www.amd.com/en/corporate/university-program.html) 